In [1]:
import modules.data as d
import modules.utils as u
from pathlib import Path

#### init ####
dataset_dir = Path('/home/mv18gs/Documents/GitHub/pathway_model/datasets/')
device = u.Devices().auto_set_device()#drop=['cuda:4','cuda:7'])

('cuda:4', 'NVIDIA A100-SXM4-80GB', 81043)
('cuda:2', 'NVIDIA A100-SXM4-80GB', 81041)
('cuda:3', 'NVIDIA A100-SXM4-80GB', 81041)
('cuda:7', 'NVIDIA A100-SXM4-80GB', 81041)
('cuda:0', 'NVIDIA A100-SXM4-80GB', 78521)
('cuda:5', 'NVIDIA A100-SXM4-80GB', 74623)
('cuda:1', 'NVIDIA A100-SXM4-80GB', 69173)
('cuda:6', 'NVIDIA A100-SXM4-80GB', 63527)

# #### Device() ####
# device = cuda:4



In [2]:
#### data ####
brca = d.TCGA(
    tcga_project = 'BRCA',
    tcga_dir = dataset_dir/'tcga',
    # type_col = 'sample_type',
    subtype_col = 'paper_BRCA_Subtype_PAM50',
    drop = ['Normal', 'Primary Tumor', 'Metastatic'],
    gene_name_path = dataset_dir/'other'/'name2ensg.csv',
    keep_noname = False,
)

kegg = d.KEGG(
    relation_filepath=dataset_dir/'other'/'relation_ohe.csv', 
    counts_data=brca,
)

dataset = d.GraphDataset(brca, kegg, kegg)
# _batch = d.get_toy_databatch(dataset, device)

# #### KEGG() ####
# _orig_kwargs             5                        dict
# relation                 (75939, 19)              DataFrame
# ensg                     4373                     list
# pathway_labels           305                      list
# edge_index               (2, 32464)               Tensor (cuda:4)
# edge_attr                (32464, 16)              Tensor (cuda:4)
# edge_labels              16                       list
# pathway_index            (4373, 305)              Tensor (cuda:4)

# #### TCGA() ####
# _orig_kwargs             9                        dict
# counts_path              PosixPath
# metadata_path            PosixPath
# gene_name_path           PosixPath
# metadata_complete        (1231, 93)               DataFrame
# metadata                 (1172, 2)                DataFrame
# y                        (1172,)                  Tensor (cuda:4)
# y_labels                 5                        list
# ensgv                    (60660, 3)             

---

* 4: node vs. set, mlp vs. gcn vs. gat
    * similar to grid2, without post-encoder mlp, with VAE & NBLoss

In [3]:
from modules.layers import AttentionSetPooling, MultiheadSetPooling
from modules.loss import MultiLoss, NBLoss, KLDLoss
from modules.model import CountsAutoencoder
from modules.norm import RawCounts, LogCounts, NBVST
from modules.train import Experiment, grid, kwarg_grid
from modules.trainers import ReconstrTrainer
from functools import partial

import torch.nn as nn
from torch_geometric.nn import GCNConv, GATConv

In [4]:
## Model

model_template = partial(
    CountsAutoencoder,
    dataset = dataset,
    # out_dim = num_nodes (default)
    embed_dim = 128,
    # head_dim = None (default)
    # num_heads = 1 (default)
    # method = 'node', !!! handled in grid !!!

    # layers
    norm_class = LogCounts,
    # encoder_class = nn.Linear !!! handled in grid !!!,
    pooling_class = AttentionSetPooling,
    mlp = False,
    variational = True,
    # out_module = nn.Linear (default)

    # layer params
    hidden_dims = 1,
    act_fn = nn.ReLU, 
    norm_fn = 'layer', 
    end_fn = False,

    # kwargs
    norm_kwargs = {'libnorm':True, 'znorm':True, 'learnable':True,}
    # pooling_kwargs = None (default)
)

model_grid = grid(
    model_template,
    method = ['node', 'set'],
    encoder_class = [nn.Linear, GCNConv, GATConv]
)

## Trainer

trainer_template = partial(
    ReconstrTrainer,
    lr = 1e-3,
    # pos_keys = None (default)
    kw_keys=('x','x_pred','theta','z_mu','z_logvar'),
    metric_keys={'pred':'x_pred', 'target':'x'},
    loss_class = MultiLoss,
    loss_kwargs = {
        'loss_classes': [NBLoss, KLDLoss],
        'loss_weights': [1.0, 1e-4],
        'loss_inputs': [
            {'x':'x', 'mu':'x_pred', 'theta':'theta'},
            {'mu':'z_mu', 'logvar':'z_logvar'}
        ],
    },
    # loss_kwargs = None (default)
    # optim_class = torch.optim.Adam (default)
    # optim_kwargs = None (default)
    early_stop = True,
    # stop_metric = 'loss', (default)
    # stop_kwargs = None (default)
    trainer_norm_class=LogCounts,
    # trainer_norm_kwargs = None (default)
)


trainer_grid = grid(
    trainer_template,
)

## Experiment

expt = Experiment(
    num_trials=25, # scVI: 5 trials
    num_epochs=300, # scVI: 200 epochs
    dataset=dataset,
    device=device,
    batch_size=128
)

expt.add_grid(
    model_grid=model_grid,
    trainer_grid=trainer_grid
)

print(len(expt.configs))

6


In [5]:
expt.configs

{'methodNode_encoderclassLinear': <modules.trainers.ReconstrTrainer at 0x7fd0041345c0>,
 'methodNode_encoderclassGCNConv': <modules.trainers.ReconstrTrainer at 0x7fd0041345c0>,
 'methodNode_encoderclassGATConv': <modules.trainers.ReconstrTrainer at 0x7fd0041345c0>,
 'methodSet_encoderclassLinear': <modules.trainers.ReconstrTrainer at 0x7fd0041345c0>,
 'methodSet_encoderclassGCNConv': <modules.trainers.ReconstrTrainer at 0x7fd0041345c0>,
 'methodSet_encoderclassGATConv': <modules.trainers.ReconstrTrainer at 0x7fd0041345c0>}

In [6]:
expt.run_experiment(
    'grid_4a_NBreconstr_gnn_set',
    report_metrics=['loss','kld','nb','mae'],
    save_csv=True,
    save_params=True,
    save_values=True,
    verbose=False,
)

  0%|          | 0/25 [00:00<?, ?it/s]

  0%|          | 0/6 [00:00<?, ?it/s]

  0%|          | 0/300 [00:00<?, ?it/s]

  0%|          | 0/300 [00:00<?, ?it/s]

  0%|          | 0/300 [00:00<?, ?it/s]

...

  0%|          | 0/300 [00:00<?, ?it/s]

  0%|          | 0/300 [00:00<?, ?it/s]

  0%|          | 0/300 [00:00<?, ?it/s]

  0%|          | 0/300 [00:00<?, ?it/s]

  0%|          | 0/300 [00:00<?, ?it/s]